# Jacob EDA

In [ ]:
# import libraries
import io
import os
from pathlib import Path

import boto3
import zstandard as zstd
from dotenv import load_dotenv, find_dotenv

import pandas as pd
import numpy as np
import time
import re

In [ ]:
from dotenv import find_dotenv, dotenv_values

env_file = find_dotenv(usecwd=True)
print("found:", env_file)
print("variables:", list(dotenv_values(env_file).keys()))

In [ ]:
# load aws credentials from .env file
load_dotenv(find_dotenv(usecwd=True), override=True)

s3_path = os.getenv("S3_PATH")
if not s3_path:
    raise RuntimeError("S3_PATH is not set — add `S3_PATH=s3://<bucket>/<prefix>/` on its own line in .env")

session = boto3.Session(profile_name=os.getenv("AWS_PROFILE"), region_name="us-east-2")
s3 = session.client("s3")
BUCKET, _, PREFIX = s3_path.strip().removeprefix("s3://").partition("/")

resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)
for obj in resp.get("Contents", []):
    print(obj["Key"], round(obj["Size"] / 1e6, 1), "MB")

In [ ]:
# Convert datasets to dataframes
# Loads from the parquet cache in DATA_DIR when it exists; otherwise parses the JSON once and saves parquet.
# Delete a parquet file to force a rebuild from S3.
DATA_DIR = Path("../data")  # project_root/data (gitignored)
DATA_DIR.mkdir(exist_ok=True)

def fetch(key):
    """Download an S3 object to DATA_DIR once; return the local path."""
    local = DATA_DIR / Path(key).name
    if not local.exists():
        s3.download_file(BUCKET, key, str(local))
    return local

def stream_jsonl(key, nrows=None, chunksize=None):
    """Read a .jsonl.zst straight from S3 without downloading the whole file."""
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"]
    reader = zstd.ZstdDecompressor(max_window_size=2**31).stream_reader(body)
    text = io.TextIOWrapper(reader, encoding="utf-8")
    return pd.read_json(text, lines=True, nrows=nrows, chunksize=chunksize)

def load_cached(parquet_path, build):
    """Read parquet if cached; otherwise call build(), save it as parquet, and return it."""
    if parquet_path.exists():
        print("loading cached:", parquet_path.resolve())
        return pd.read_parquet(parquet_path)
    print("building (slow, first run only):", parquet_path.name)
    df = build()
    df.to_parquet(parquet_path, engine="pyarrow", index=False)
    print("saved:", parquet_path.resolve(), round(parquet_path.stat().st_size / 1e6, 1), "MB")
    return df

# companies: ~252 MB compressed — small enough to download and load fully
companies_raw = load_cached(
    DATA_DIR / "companies_2026_09_30.parquet",
    lambda: pd.read_json(fetch(PREFIX + "companies_2026_09_30.jsonl.zst"), compression="zstd", lines=True),
)

# job_openings: ~19 GB compressed — too big for memory; sample it for EDA
# (for full-file stats, use chunksize=100_000 and aggregate per chunk)
job_openings_sample = load_cached(
    DATA_DIR / "job_openings_sample_2026_09_30.parquet",
    lambda: stream_jsonl(PREFIX + "job_openings_2026_09_30.jsonl.zst", nrows=200_000),
)

print("companies_raw shape:", companies_raw.shape)
print("job_openings_sample shape:", job_openings_sample.shape)

In [ ]:
# Inspect structure: column names and the Python type of each value in the first row
for name, df in [("companies_raw", companies_raw), ("job_openings_sample", job_openings_sample)]:
    print(name, list(df.columns))
    print(df.iloc[0].map(lambda v: type(v).__name__).to_dict())

In [ ]:
# Flatten JSON:API documents: each row holds a list of records in `data`
# and a list of related objects in `included` (referenced via relationships)
JOBS_N = 20_000  # job-opening documents to flatten; raise once this runs comfortably

def unpack(df, col, dedupe=False):
    """One row per record across all documents, nested dicts flattened to dotted columns.
    dedupe=True keeps the first (type, id) only, *before* flattening, to save memory."""
    start = time.time()
    seen, records = set(), []
    for batch in df[col]:
        # lists come back as numpy arrays when loaded from the parquet cache
        if not isinstance(batch, (list, np.ndarray)):
            continue
        for r in batch:
            if dedupe:
                key = (r.get("type"), r.get("id"))
                if key in seen:
                    continue
                seen.add(key)
            records.append(r)
    print(f"  {col}: {len(records):,} records collected, flattening...", flush=True)
    out = pd.json_normalize(records)
    print(f"  {col}: {out.shape} in {time.time() - start:.0f}s", flush=True)
    return out

print("companies", flush=True)
companies = unpack(companies_raw, "data")
companies_included = unpack(companies_raw, "included", dedupe=True)

print(f"job_openings (first {JOBS_N:,} documents)", flush=True)
jobs_docs = job_openings_sample.head(JOBS_N)
job_openings = unpack(jobs_docs, "data")
job_openings_included = unpack(jobs_docs, "included", dedupe=True)

for name, df in [("companies", companies), ("companies_included", companies_included),
                 ("job_openings", job_openings), ("job_openings_included", job_openings_included)]:
    print(f"{name}: {df.shape}")
    print("  ", list(df.columns)[:25], "..." if df.shape[1] > 25 else "")

In [ ]:
# Write a df.info()-style schema for each flattened table into PROJECT_PLAN.md
# Column names, non-null counts and dtypes only, never data values
PLAN_PATH = DATA_DIR.parent / "PROJECT_PLAN.md"

def schema_md(name, df, note=""):
    non_null = df.notna().sum()
    lines = [
        f"#### `{name}`: {len(df):,} rows × {df.shape[1]} columns{note}",
        "",
        "| # | Column | Non-null | Null % | Dtype |",
        "|---|---|---|---|---|",
    ]
    for i, (col, dtype) in enumerate(df.dtypes.items()):
        null_pct = 100 * (1 - non_null[col] / len(df)) if len(df) else 0
        lines.append(f"| {i} | `{col}` | {non_null[col]:,} | {null_pct:.1f}% | {dtype} |")
    return "\n".join(lines)

jobs_note = f" (from the first {JOBS_N:,} job-opening documents)"
sections = [
    schema_md("companies", companies),
    schema_md("companies_included", companies_included, " (duplicates removed by type and id)"),
    schema_md("job_openings", job_openings, jobs_note),
    schema_md("job_openings_included", job_openings_included, jobs_note + ", duplicates removed by type and id"),
]
generated = f"_Generated {pd.Timestamp.now():%Y-%m-%d %H:%M} from the 2026-09-30 snapshot._\n\n" + "\n\n".join(sections)

plan = PLAN_PATH.read_text()
pattern = r"(<!-- schema:start -->\n).*?(\n<!-- schema:end -->)"
if not re.search(pattern, plan, flags=re.S):
    raise RuntimeError("schema markers not found in PROJECT_PLAN.md")
PLAN_PATH.write_text(re.sub(pattern, lambda m: m.group(1) + generated + m.group(2), plan, flags=re.S))
print("wrote schemas for", len(sections), "tables to", PLAN_PATH.resolve())